# Multivariate Time Series

In [1]:
import pandas as pd
import tensorflow as tf

In [2]:
df = pd.read_csv(r'D:\Latihan\Deep-Learning\dataset\household_power_consumption.csv', sep=',', parse_dates=['datetime'], index_col='datetime', header=0)
df

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...
2007-02-14 17:19:00,0.636,0.140,241.16,2.6,0.0,0.0,0.0
2007-02-14 17:20:00,0.552,0.000,240.46,2.2,0.0,0.0,0.0
2007-02-14 17:21:00,0.538,0.000,239.74,2.2,0.0,0.0,0.0


In [3]:
df_nm = len(df.columns)
print(df_nm)

7


In [4]:
def normalize_series(data, min, max):
    data = data - min
    data = data / max
    return data

data = df.values
data = normalize_series(data, data.min(axis=0), data.max(axis=0))
print(data.shape)

(86400, 7)


In [5]:
N_FEATURES = len(df.columns)
print(N_FEATURES)

7


In [6]:
SPLIT_TIME = int(len(data) * 0.5)
x_train = data[:SPLIT_TIME]
x_valid = data[SPLIT_TIME:]

In [7]:
def windowed_dataset(series, batch_size, n_past=24, n_future=24, shift=1):
    ds = tf.data.Dataset.from_tensor_slices(series)
    ds = ds.window(size=n_past + n_future, shift=shift, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(n_past + n_future))
    ds = ds.map(lambda w: (w[:n_past], w[n_past:]))
    return ds.batch(batch_size).prefetch(1)

In [8]:
BATCH_SIZE = 32
N_PAST = 24
N_FUTURE = 24
SHIFT = 1
# Kode untuk membuat windowed datasets
train_set = windowed_dataset(series=x_train, batch_size=BATCH_SIZE, n_past=N_PAST, n_future=N_FUTURE, shift=SHIFT)
valid_set = windowed_dataset(series=x_valid, batch_size=BATCH_SIZE, n_past=N_PAST, n_future=N_FUTURE, shift=SHIFT)

In [9]:
model = tf.keras.models.Sequential([
        tf.keras.layers.Dense(64, input_shape=(N_PAST, N_FEATURES)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(N_FEATURES)
    ])

d:\Latihan\Deep-Learning\env_deep-learning\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
class myCallback(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs={}):
            if (logs.get('mae') < 0.055 and logs.get('val_mae') < 0.055):
                self.model.stop_training = True
 
callbacks = myCallback()

In [13]:
# Kode untuk melakukan menyusun struktur sesuai dengan machine learning
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3) 
model.compile(loss='mae',
                  optimizer= optimizer,
                  metrics=["mae"])

In [14]:
model.fit(train_set,
          validation_data=(valid_set),
          epochs=100,
          callbacks=callbacks,
          verbose=1
    )

Epoch 1/100


d:\Latihan\Deep-Learning\env_deep-learning\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


   1341/Unknown 8s 5ms/step - loss: 0.0697 - mae: 0.0697

d:\Latihan\Deep-Learning\env_deep-learning\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1349/1349 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0696 - mae: 0.0696 - val_loss: 0.0590 - val_mae: 0.0590
Epoch 2/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - loss: 0.0598 - mae: 0.0598 - val_loss: 0.0607 - val_mae: 0.0607
Epoch 3/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 0.0593 - mae: 0.0593 - val_loss: 0.0592 - val_mae: 0.0592
Epoch 4/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - loss: 0.0588 - mae: 0.0588 - val_loss: 0.0626 - val_mae: 0.0626
Epoch 5/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - loss: 0.0584 - mae: 0.0584 - val_loss: 0.0574 - val_mae: 0.0574
Epoch 6/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0581 - mae: 0.0581 - val_loss: 0.0588 - val_mae: 0.0588
Epoch 7/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0579 - mae: 0.0579 - val_loss: 0.0579 - val_mae: 0.0579
Epoch 8/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0577 - mae: 0.0577 - val_loss: 0.0575 - val_mae: 0.0575
Epoch 9/100
1349/1349 ━━━━━━━━━━━━

In [15]:
train_pred = model.predict(train_set)
train_pred[0][0]

1349/1349 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step


array([ 0.3491039 ,  0.22011213,  0.03417588,  0.33925354, -0.01061149,
        0.00541351,  0.847354  ], dtype=float32)